#  RoBERTa Classifier with SupCon Pretrained Encoder

## Hyperparams and settings

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
from settings import Hyperparams, Settings

SETTINGS_PATH = "../configs/settings.yaml"
HYPERPARAMS_PATH = "../configs/hyperparams.yaml"

SETTINGS = Settings(yaml_path=SETTINGS_PATH)
HYPERPARAMS = Hyperparams(yaml_path=HYPERPARAMS_PATH)

In [ ]:
import torch

device = torch.device(SETTINGS.device)

## Tokenizer

In [ ]:
from transformers import XLMRobertaTokenizer

tokenizer = XLMRobertaTokenizer.from_pretrained(SETTINGS.pretrained_encoder_directory)

## Data

### Training Data

In [ ]:
import pandas as pd

train_dataframe = pd.read_csv(SETTINGS.train_dataset_path)

In [ ]:
from torch.utils.data import DataLoader

from cleaner import Cleaner
from collate_function import collate_function
from dataset import VacanciesDataset

cleaner = Cleaner()

train_dataset = VacanciesDataset(train_dataframe, cleaner.process)
train_loader = DataLoader(
    train_dataset,
    batch_size=HYPERPARAMS.batch_size,
    collate_fn=lambda data: collate_function(data, tokenizer=tokenizer),
    shuffle=True,
)

### Validation Data

In [ ]:
import pandas as pd

valid_dataframe = pd.read_csv(SETTINGS.valid_dataset_path)

In [ ]:
valid_dataset = VacanciesDataset(valid_dataframe, cleaner.process)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=HYPERPARAMS.batch_size,
    collate_fn=lambda data: collate_function(data, tokenizer=tokenizer),
    shuffle=True,
)

## Model

### Encoder

In [ ]:
from transformers import XLMRobertaConfig, XLMRobertaModel

config = XLMRobertaConfig.from_pretrained("xlm-roberta-base")

encoder = XLMRobertaModel(config)
encoder.resize_token_embeddings(len(tokenizer))

state_dict = torch.load(f"{SETTINGS.pretrained_encoder_directory}/state_dict.pth")
encoder.load_state_dict(state_dict)

In [ ]:
from freezing import freeze_embeddings, freeze_layers_below

freeze_layers_below(encoder, HYPERPARAMS.freeze_layers)

if HYPERPARAMS.freeze_embeddings:
    freeze_embeddings(encoder)

#### Memory optimization

In [ ]:
encoder.config.use_cache = False
encoder.gradient_checkpointing_enable()

### Classifier

In [ ]:
from model import RobertaClassifier

NUMBER_OF_CLASSES = 3

model = RobertaClassifier(encoder=encoder, num_classes=NUMBER_OF_CLASSES).to(device)

## Loss and Optimizer

In [ ]:
STAFF_TYPE_MAP = {
    "class-0": 0,
    "class-1": 1,
    "class-2": 2,
}

value_counts = train_dataframe["staff_type"].value_counts()
total_objects = len(train_dataframe)

weights = []

for staff_type in STAFF_TYPE_MAP:
    staff_type_objects = value_counts[staff_type]

    weight = total_objects / (NUMBER_OF_CLASSES * staff_type_objects)
    weights.append(float(weight))

weights = torch.tensor(weights, dtype=torch.float32).to(device)

In [ ]:
from torch.nn import CrossEntropyLoss

criterion = CrossEntropyLoss(weight=weights)

In [ ]:
from torch import optim

optimizer = optim.AdamW(
    model.parameters(),
    lr=HYPERPARAMS.learning_rate,
    weight_decay=HYPERPARAMS.weight_decay,
)

## Scheduler

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

scheduler = CosineAnnealingLR(optimizer, T_max=HYPERPARAMS.epochs, eta_min=HYPERPARAMS.eta_min)

## Memory warmup

## Train

In [ ]:
import mlflow

mlflow.set_tracking_uri(SETTINGS.tracking_uri)
mlflow.set_experiment(SETTINGS.experiment_name)

if mlflow.active_run():
    mlflow.end_run()

run = mlflow.start_run(run_name=SETTINGS.run_name)

mlflow.log_params(vars(HYPERPARAMS))

In [ ]:
from tqdm.auto import tqdm

from training import train_step
from validation import valid_step

total_epochs = HYPERPARAMS.epochs
global_step = 0

for epoch in tqdm(range(total_epochs)):
    epoch_train_loss = 0.0
    epoch_valid_loss = 0.0

    for batch_X, batch_y in train_loader:
        output = train_step(model, optimizer, criterion, batch_X, batch_y)

        epoch_train_loss += output["loss"]

        mlflow.log_metrics(
            {
                "loss": output["loss"],
                "gradient norm": output["grad_norm"],
            },
            step=global_step,
        )

        global_step += 1

    del output

    current_lr = optimizer.param_groups[0]["lr"]
    mlflow.log_metric("learning rate", current_lr, epoch)
    scheduler.step()

    for batch_X, batch_y in valid_loader:
        output = valid_step(model, criterion, batch_X, batch_y)
        epoch_valid_loss += output["loss"]
        del output

    train_loss = epoch_train_loss/len(train_loader)
    valid_loss = epoch_valid_loss/len(valid_loader)

    mlflow.log_metrics({
        "learning rate": current_lr,
        "average train loss": train_loss,
        "average valid loss": valid_loss,
    }, epoch)

    avg_loss = epoch_train_loss / len(train_loader)
    print(f"Epoch [{epoch + 1}/{HYPERPARAMS.epochs}] Train loss: {train_loss} | Validation loss {valid_loss}")

## Validation

In [ ]:
from validation import valid_step

all_preds = []
all_labels = []

model.eval()
for batch_X, batch_y in valid_loader:
    output = valid_step(model, criterion, batch_X, batch_y)

    preds = torch.argmax(output["logits"], dim=-1)

    all_preds.append(preds.cpu())
    all_labels.append(batch_y.cpu())

all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

## Log Classification Report

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(all_labels, all_preds, target_names=["class-0", "class-1", "class-2"], output_dict=True)

report_df = pd.DataFrame(report).transpose().reset_index()
report_df = report_df.rename(columns={"index": "category"})

mlflow.log_table(data=report_df, artifact_file="classification_report.json")

## Log Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues)
plt.title("Evaluation Confusion Matrix")

fig = plt.gcf()
mlflow.log_figure(fig, "eval_confusion_matrix.png")

In [ ]:
mlflow.end_run()

## Save

In [ ]:
from pathlib import Path

torch.save(model.state_dict(), Path(SETTINGS.model_directory) / "state_dict.pth")

## Clear memory

In [ ]:
import gc

del model

gc.collect()
torch.cuda.empty_cache()